# Loading Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

# Embeddings
from sentence_transformers import SentenceTransformer

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Miscellaneous
import warnings
warnings.filterwarnings('ignore')

# NLTK downloads
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
# For working on Google Colab 
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')

# Loading files 
train_df_train = pd.read_csv("/content/drive/MyDrive/fakenews/train_df_train.csv")
train_df_val = pd.read_csv("/content/drive/MyDrive/fakenews/train_df_val.csv")
therealtest_df = pd.read_csv("/content/drive/MyDrive/fakenews/therealtest_df.csv")

# Cleaning up this is for loading the csv files 
train_df_train['text'] = train_df_train['text'].fillna('').astype(str)
train_df_val['text'] = train_df_val['text'].fillna('').astype(str)
therealtest_df['text'] = therealtest_df['text'].fillna('').astype(str)

# Exploratory Data Analysis (EDA)

In [ ]:
# Loading datasets
train_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Project Info/dataset/training_data.csv", delimiter = "\t", header = None)
print("Training Dataset Shape:", train_df.shape)

# Renaming for clarity
train_df.columns = ['label', 'text']

# Missing values check
print("Missing Values in Training Dataset:", train_df.isnull().sum())

# Duplicates check 
print("Duplicate Rows in Training Dataset:", train_df.duplicated().sum())

# Removing duplicates
train_df.drop_duplicates(inplace=True)
print("\nShape After Removing Duplicates:")
print("Training:", train_df.shape)

## Data Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower() # make everything lower case
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text) # removing punctuation
    text = re.sub(r'\d+', '', text) # remove numbers
    text = re.sub(r'\s+', ' ', text) # remove extra spaces
    words = text.split() # tokenize
    words = [ # remove stopwords + lemmatize
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [ ]:
train_df['text'] = train_df['text'].apply(clean_text)
print("Training after cleaning", train_df.head(10))

# Data Cleaning for the Real Test Dataset 
* This is for the dataset (placeholder = 2) where we want to test our trained models
* It has to go through all the data cleaning we have done for our train/val

In [ ]:
therealtest_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Project Info/dataset/testing_data.csv", delimiter = "\t", header = None)
therealtest_df.columns = ['label', 'text']
print("Missing Values in Testing Dataset:", therealtest_df.isnull().sum())
print("Duplicate Rows in Testing Dataset:", therealtest_df.duplicated().sum())
therealtest_df.drop_duplicates(inplace=True)

print("\nShape After Removing Duplicates:")
print("Testing:", therealtest_df.shape)

therealtest_df['text'] = therealtest_df['text'].apply(clean_text)
print("Test after cleaning", therealtest_df.head(10))

# Test-train Splitting
* This one is just from the training_data

In [ ]:
train_df_train, train_df_val = train_test_split(train_df, test_size = 0.2, random_state = 42, stratify = train_df['label'])

In [ ]:
# Saving training and test dataset 80-20
train_df_train.to_csv("train_df_train.csv", index = False)
train_df_val.to_csv("train_df_val.csv", index = False)

# Part 1 - Classical NLP
* Bag of Words (BOW)
* Term Frequency-Inverse Document Frequency (TF-IDF)
### The Classifier for classical NLP
* Multinomial Naive Bayes (MultinomialNB)

#### Bag of Words

In [ ]:
vectorizer = CountVectorizer()

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])

# Print shape of vectorized dataset
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

#### TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()

# Vectorize all dataset
train_df_train_tfidf = vectorizer.fit_transform(train_df_train['text'])
train_df_val_tfidf = vectorizer.transform(train_df_val['text'])

# Print shape of vectorized dataset
print("Training TF-IDF Shape:", train_df_train_tfidf.shape)
print("Validation TF-IDF Shape:", train_df_val_tfidf.shape)

#### Multinomial Naive Bayes (MultinomialNB)

In [ ]:
model = MultinomialNB()

## Fine-tuning of BOW 

#### First we run default BOW and default MultinomialNB

In [ ]:
model.fit(train_df_train_bow, train_df_train['label'])
y_pred = model.predict(train_df_val_bow)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### The hyperparameters that are tuned for BOW:
* ngram_range
* max_df
* min_df
* max_features

#### Tuning #1:

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_df=0.95,
    min_df=2,
    max_features=10000
)

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

model.fit(train_df_train_bow, train_df_train['label'])
y_pred = model.predict(train_df_val_bow)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)


#### Tuning #2:

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_df=0.95,
    min_df=2,
    max_features=10000
)

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

model.fit(train_df_train_bow, train_df_train['label'])
y_pred = model.predict(train_df_val_bow)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### Tuning #3:

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_df= 0.99,
    min_df= 1,
    max_features= 50000
)

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)


model.fit(train_df_train_bow, train_df_train['label'])
y_pred = model.predict(train_df_val_bow)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### Tuning #4:

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_df= 0.95,
    min_df= 2,
    max_features= 50000
)

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

model.fit(train_df_train_bow, train_df_train['label'])
y_pred = model.predict(train_df_val_bow)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

## Fine-tuning of TF-IDF

In [ ]:
model.fit(train_df_train_tfidf, train_df_train['label'])
y_pred = model.predict(train_df_val_tfidf)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### The hyperparameters that are tuned for TF-IDF:
* ngram_range
* max_df
* min_df
* max_features
* sublinear_tf

#### Tuning #1:

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range = (1,2),
    max_df = 0.95,
    min_df = 2,
    max_features=20000,
    sublinear_tf=True
)

train_df_train_tfidf = vectorizer.fit_transform(train_df_train['text'])
train_df_val_tfidf = vectorizer.transform(train_df_val['text'])
print("Training TF-IDF Shape:", train_df_train_tfidf.shape)
print("Validation TF-IDF Shape:", train_df_val_tfidf.shape)

model.fit(train_df_train_tfidf, train_df_train['label'])
y_pred = model.predict(train_df_val_tfidf)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### Tuning #2:

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range = (1,3),
    max_df = 0.95,
    min_df = 2,
    max_features=50000,
    sublinear_tf=True
)

train_df_train_tfidf = vectorizer.fit_transform(train_df_train['text'])
train_df_val_tfidf = vectorizer.transform(train_df_val['text'])
print("Training TF-IDF Shape:", train_df_train_tfidf.shape)
print("Validation TF-IDF Shape:", train_df_val_tfidf.shape)

model.fit(train_df_train_tfidf, train_df_train['label'])
y_pred = model.predict(train_df_val_tfidf)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

# Part 2: Embedding
## Sentence Transformers
* Model: all-MiniLM-L6-v2
* Architecture: MiniLM (a smaller, faster variant of BERT).
### Classifier of choice: 
* LogisticRegression
* LinearSVC

In [ ]:
pip install sentence-transformers

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

#### Converting text to embeddings

In [ ]:
train_df_train_embed = embedding_model.encode(train_df_train['text'].tolist())
train_df_val_embed = embedding_model.encode(train_df_val['text'].tolist())

##### Classifier: Logistic Regression

In [ ]:
model = LogisticRegression(
    max_iter = 1000,
    random_state = 42
)

In [ ]:
model.fit(train_df_train_embed, train_df_train['label'])
y_pred = model.predict(train_df_val_embed)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### The hyperparameters that are tuned for LogisticRegression:
* C
* max_iter

#### Fine-tuning #1:

In [ ]:
model = LogisticRegression(
    C = 10,
    max_iter = 1000,
    random_state = 42
)

model.fit(train_df_train_embed, train_df_train['label'])
y_pred = model.predict(train_df_val_embed)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

##### Classifier: LinearSVC

In [ ]:
model = LinearSVC(random_state = 42)

In [ ]:
model.fit(train_df_train_embed, train_df_train['label'])
y_pred = model.predict(train_df_val_embed)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### The hyperparameters that are tuned for LinearSVC:
* C

#### Fine-tuning #1:

In [ ]:
model = LinearSVC(
    C = 10,
    random_state = 42)

model.fit(train_df_train_embed, train_df_train['label'])
y_pred = model.predict(train_df_val_embed)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

#### Fine-tuning #2:

In [ ]:
model = LinearSVC(
    C = 100,
    random_state = 42)

model.fit(train_df_train_embed, train_df_train['label'])
y_pred = model.predict(train_df_val_embed)
matrix = confusion_matrix(train_df_val['label'], y_pred)
print(classification_report(train_df_val['label'], y_pred))
print(matrix)

# Part 3: Transformer Model
### DistilBERT (destilbert-base-uncased)

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
train_dataset = Dataset.from_pandas(train_df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(train_df_val[['text', 'label']])

In [ ]:
model_name = "distilbert-base-uncased"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):

  return tokenizer(
      example['text'],
      trauncation = True,
      padding = 'max_length',
      max_length = 256
  )

train_dataset = train_dataset.map(tokenize_function, batched = True)
val_dataset = val_dataset.map(tokenize_function, batched = True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2
)

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  precision, recall, f1, _ = precision_recall_fscore_support(
      labels,
      predictions,
      average = 'binary'
  )

  accuracy = accuracy_score(labels, predictions)

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1
  }

training_args = TrainingArguments(
    output_dir = "./results",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 2,
    weight_decay = 0.01,
    logging_dir = "./logs",
    logging_steps = 100,
    load_best_model_at_end = True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

# Part 4: Inference on Unseen Test Dataset
* Now we are testing from the unseen dataset
* therealtest_df (that has gone through the same process of data cleaning)


## Real test #1: BOW + MultinomialNB

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_df= 0.99,
    min_df= 1,
    max_features= 50000
)

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

model = MultinomialNB()
model.fit(train_df_train_bow, train_df_train['label'])

real_test_bow = vectorizer.transform(therealtest_df['text'])
real_predictions = model.predict(real_test_bow)
therealtest_df['label'] = real_predictions
print(therealtest_df.head())

therealtest_df.to_csv("bow_multinomial_predictions.csv")
files.download("bow_multinomial_predictions.csv")


## Real test #2: TF-IDF + MultinomialNB

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range = (1,3),
    max_df = 0.95,
    min_df = 2,
    max_features=50000,
    sublinear_tf=True
)

train_df_train_tfidf = vectorizer.fit_transform(train_df_train['text'])
train_df_val_tfidf = vectorizer.transform(train_df_val['text'])
print("Training TF-IDF Shape:", train_df_train_tfidf.shape)
print("Validation TF-IDF Shape:", train_df_val_tfidf.shape)

model = MultinomialNB()
model.fit(train_df_train_tfidf, train_df_train['label'])

real_test_tfidf = vectorizer.transform(therealtest_df['text'])
real_predictions = model.predict(real_test_tfidf)
therealtest_df['label'] = real_predictions
print(therealtest_df.head())
therealtest_df.to_csv("tfidf_multinomial_predictions.csv")

files.download("tfidf_multinomial_predictions.csv")



## Real test #3: Embedding (Sentence Transformers) + LogisticRegression

In [ ]:
therealtest_embed = embedding_model.encode(therealtest_df['text'].tolist())
embed_lr_predictions = model.predict(therealtest_embed)
therealtest_df['label'] = embed_lr_predictions
print(therealtest_df.head())
therealtest_df.to_csv("embedding_lr_predictions.csv")

files.download("embedding_lr_predictions.csv")

## Real test #4: Embedding (Sentence Transformers) + LinearSVC

In [ ]:
therealtest_embed = embedding_model.encode(therealtest_df['text'].tolist())
embed_lr_predictions = model.predict(therealtest_embed)
therealtest_df['label'] = embed_lr_predictions
print(therealtest_df.head())
therealtest_df.to_csv("embedding_linearsvc_predictions.csv")

files.download("embedding_linearsvc_predictions.csv")

## Real test #5: DistilBERT

In [ ]:
real_test_dataset = Dataset.from_pandas(therealtest_df[['text']])
real_test_dataset = real_test_dataset.map(tokenize_function, batched = True) # tokenize
predictions = trainer.predict(real_test_dataset) # predict
predicted_labels = np.argmax(predictions.predictions, axis = 1) # Convert logits to labels
therealtest_df['label'] = predicted_labels
print(therealtest_df.head())
therealtest_df.to_csv("distilbert_predictions.csv")

files.download("distilbert_predictions.csv")

# Visualization

In [ ]:
bow_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Submission/bow_multinomial_predictions.csv")
tfidf_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Submission/tfidf_multinomial_predictions.csv")
embed_lr_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Submission/embedding_lr_predictions.csv")
embed_svc_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Submission/embedding_linearsvc_predictions.csv")
distilbert_df = pd.read_csv("C:/Users/anmnu/DSML/Week7/Project/Submission/distilbert_predictions.csv")

files = {
    "BoW + NB": bow_df,
    "TF-IDF + NB": tfidf_df,
    "Embed + LR": embed_lr_df,
    "Embed + SVC": embed_svc_df,
    "DistilBERT": distilbert_df
}


# Validation accuracies from training
accuracies = {
    "BoW + NB": 0.94,
    "TF-IDF + NB": 0.94,
    "Embed + LR": 0.92,
    "Embed + SVC": 0.93,
    "DistilBERT": 0.9626
}


# Lists for plotting
model_names = []
fake_counts = []
real_counts = []
fake_percentages = []
real_percentages = []


# Calculate counts and percentages
for model_name, df in files.items():

    counts = df['label'].value_counts().sort_index()

    fake_count = counts.get(0, 0)
    real_count = counts.get(1, 0)

    total = len(df)

    fake_percent = (fake_count / total) * 100
    real_percent = (real_count / total) * 100

    model_names.append(model_name)

    fake_counts.append(fake_count)
    real_counts.append(real_count)

    fake_percentages.append(fake_percent)
    real_percentages.append(real_percent)

x = np.arange(len(model_names))
width = 0.35

plt.figure(figsize=(12,7))

bars1 = plt.bar(
    x - width/2,
    fake_counts,
    width,
    label='Fake News (0)'
)

bars2 = plt.bar(
    x + width/2,
    real_counts,
    width,
    label='Real News (1)'
)


# Add percentage labels above bars
for i, bar in enumerate(bars1):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 40,
        f"{fake_percentages[i]:.1f}%",
        ha='center'
    )


for i, bar in enumerate(bars2):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 40,
        f"{real_percentages[i]:.1f}%",
        ha='center'
    )


# Add accuracy below x-axis
for i, model in enumerate(model_names):

    plt.text(
        x[i],
        -500,
        f"Acc: {accuracies[model]:.3f}",
        ha='center',
        fontsize=10
    )


plt.xticks(x, model_names)
plt.ylabel("Number of Predictions")
plt.title("Prediction Distribution Across Models")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()